# 06. Baseline Modeling

## Objective

This notebook begins the modeling phase. The first goal is not to train a complex algorithm. It is to establish a correct temporal evaluation procedure and measure the performance of the transparent business rule `RecencyDays >= 30`. Every later model must be evaluated on the same validation observations and must improve on this reference in a useful way. The final test period remains untouched.

## Work plan

1. Load the point-in-time feature table and temporal fold definitions.
2. Separate development observations from the locked final test.
3. Reconstruct every purged training and validation fold.
4. Evaluate the deterministic recency baseline on validation only.
5. Use these results as the minimum benchmark for the first trained model.

## 1. Load modeling inputs

In [ ]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

modeling_data = pd.read_parquet(
    "../data/processed/churn_snapshot_features.parquet"
)
fold_date_columns = ["TrainStart", "ValidationStart", "ValidationEnd"]
model_selection_folds = pd.read_csv(
    "../data/interim/model_selection_folds.csv",
    parse_dates=fold_date_columns,
)

print("Modeling data:", modeling_data.shape)
print("Fold definitions:", model_selection_folds.shape)

## 2. Modeling contract

`feature_columns` contains only variables known at the reference date. `target_column` is used only as the outcome. Identifiers, dates, churn deadlines, current status, outcome timestamps, split labels, and the target itself are never passed to a model.

In [ ]:
feature_columns = [
    "RecencyDays",
    "ObservedTenureDays",
    "PurchaseFrequency",
    "MonetaryValue",
    "AverageInvoiceValue",
    "RecencyToMedianGapRatio",
    "ChristmasInvoiceShare",
    "UniqueProductsPurchased",
    "ProductsPerInvoice",
    "IsUK",
    "HasCancellation",
    "PurchasesLast30Days",
    "IsInChurnRiskWindow",
]
target_column = "WillChurnNext30Days"

forbidden_columns = {
    "Customer ID", "ReferenceDate", "FirstObservedPurchaseDate", "Country",
    "LastPurchaseDate", "ChurnDeadline", "IsChurnedAtReference",
    "OutcomeEndDate", "ChurnEventDate", "LabelEndDate",
    "FinalSplit", target_column,
}

print("Selected features:", feature_columns)
print("Forbidden columns selected:", set(feature_columns) & forbidden_columns)

## 3. Development and locked test

Development observations may be used for model design and validation. Test observations are isolated but their target distribution and performance are not inspected. They will be used once, after the complete pipeline and decision threshold are frozen.

In [ ]:
development_data = modeling_data.loc[
    modeling_data["FinalSplit"].eq("development")
].copy()
locked_test_data = modeling_data.loc[
    modeling_data["FinalSplit"].eq("test")
].copy()

print("Development rows:", len(development_data))
print("Development reference dates:", development_data["ReferenceDate"].nunique())
print("Locked test rows:", len(locked_test_data))
print("Locked test reference dates:", locked_test_data["ReferenceDate"].nunique())

## 4. Purged temporal folds

For each fold, training snapshots occur before validation. A candidate training row is retained when `LabelEndDate <= ValidationStart`. Equality is safe because target windows exclude their end timestamp and validation features use only events strictly before their reference date. A row is purged only when its label requires information after validation has begun. The function returns the complete rows so the same split can later be reused for preprocessing and model fitting.

In [ ]:
def get_fold_data(fold):
    candidate_train = development_data["ReferenceDate"].between(
        fold["TrainStart"],
        fold["ValidationStart"],
        inclusive="left",
    )
    train_mask = (
        candidate_train
        & development_data["LabelEndDate"].le(fold["ValidationStart"])
    )
    validation_mask = development_data["ReferenceDate"].between(
        fold["ValidationStart"],
        fold["ValidationEnd"],
    )
    return (
        development_data.loc[train_mask].copy(),
        development_data.loc[validation_mask].copy(),
    )

fold_checks = []
for _, fold in model_selection_folds.iterrows():
    fold_train, fold_validation = get_fold_data(fold)
    fold_checks.append({
        "Approach": fold["Approach"],
        "Fold": fold["Fold"],
        "TrainRows": len(fold_train),
        "ValidationRows": len(fold_validation),
        "LatestTrainReference": fold_train["ReferenceDate"].max(),
        "LatestTrainLabelEnd": fold_train["LabelEndDate"].max(),
        "ValidationStart": fold_validation["ReferenceDate"].min(),
        "NoLabelOverlap": (
            fold_train["LabelEndDate"].max()
            <= fold_validation["ReferenceDate"].min()
        ),
    })
fold_checks = pd.DataFrame(fold_checks)
fold_checks

## 5. Deterministic recency baseline

This baseline does not learn anything from the training data. It predicts churn for every validation snapshot with at least 30 days of current recency. It is evaluated separately on every approved validation fold.

The most useful metrics are:

- `Precision`: among contacted customers, the proportion that actually churns.
- `Recall`: among future churners, the proportion detected.
- `Specificity`: among customers who remain active, the proportion correctly left unflagged.
- `ContactRate`: the share of the active population flagged by the rule.
- `BalancedAccuracy`: the average of recall and specificity.

In [ ]:
baseline_results = []

for _, fold in model_selection_folds.iterrows():
    _, validation = get_fold_data(fold)
    y_true = validation[target_column]
    y_pred = validation["RecencyDays"].ge(30)
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[False, True]
    ).ravel()

    baseline_results.append({
        "Approach": fold["Approach"],
        "Fold": fold["Fold"],
        "ValidationRows": len(validation),
        "ChurnRate": y_true.mean(),
        "ContactRate": y_pred.mean(),
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "Specificity": tn / (tn + fp),
        "F1": f1_score(y_true, y_pred),
        "Accuracy": accuracy_score(y_true, y_pred),
        "BalancedAccuracy": balanced_accuracy_score(y_true, y_pred),
        "TrueNegatives": tn,
        "FalsePositives": fp,
        "FalseNegatives": fn,
        "TruePositives": tp,
    })

baseline_results = pd.DataFrame(baseline_results)
percentage_columns = [
    "ChurnRate", "ContactRate", "Precision", "Recall",
    "Specificity", "F1", "Accuracy", "BalancedAccuracy",
]
baseline_display = baseline_results.copy()
baseline_display[percentage_columns] *= 100
baseline_display.round(2)

## 6. Baseline interpretation

The recency rule is a benchmark, not a trained model. On the October-November 2010 validation, it obtains 99.58% recall and 76.97% precision, with 284 false positives and 4 false negatives. On January-February 2011, it obtains 100% recall and 90.58% precision, with 149 false positives and no false negatives.

The target rate changes from 24.86% in the first validation period to 45.46% in the second. This difference explains why one validation period is insufficient. Approach A and fold 2 of Approach B intentionally produce the same baseline result because they use the same January-February validation observations; Approach B additionally measures stability on October-November.

The baseline's main weakness is false positives: customers above 30 days of recency who purchase before reaching 60 days. The first trained model must reduce these unnecessary alerts without losing too many true future churners. We begin with a one-feature logistic regression before adding the other RFM variables.

## 7. Model 1: recency-only logistic regression

### Hypothesis

The probability of reaching churn during the next 30 days increases smoothly as `RecencyDays` increases. Unlike the baseline's hard cutoff, logistic regression learns the shape and position of this transition from training data and returns a probability.

This first model intentionally uses only `RecencyDays`. If it fails, the reason is easy to understand. If it succeeds, later models must show that additional variables provide value beyond recency. `StandardScaler` and `LogisticRegression` are placed in one pipeline so scaling is learned only from each fold's training data. The classification threshold remains at the default 0.50 for this first diagnostic.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

model_1_features = ["RecencyDays"]
model_1_results = []
model_1_fitted = {}

for _, fold in model_selection_folds.iterrows():
    train, validation = get_fold_data(fold)
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, solver="liblinear"),
    )
    model.fit(train[model_1_features], train[target_column])
    model_1_fitted[(fold["Approach"], fold["Fold"])] = model

    for dataset_name, dataset in [("train", train), ("validation", validation)]:
        y_true = dataset[target_column]
        y_probability = model.predict_proba(dataset[model_1_features])[:, 1]
        y_pred = y_probability >= 0.50
        tn, fp, fn, tp = confusion_matrix(
            y_true, y_pred, labels=[False, True]
        ).ravel()

        model_1_results.append({
            "Approach": fold["Approach"],
            "Fold": fold["Fold"],
            "Dataset": dataset_name,
            "Rows": len(dataset),
            "ChurnRate": y_true.mean(),
            "ContactRate": y_pred.mean(),
            "Precision": precision_score(y_true, y_pred),
            "Recall": recall_score(y_true, y_pred),
            "Specificity": tn / (tn + fp),
            "F1": f1_score(y_true, y_pred),
            "Accuracy": accuracy_score(y_true, y_pred),
            "BalancedAccuracy": balanced_accuracy_score(y_true, y_pred),
            "PR_AUC": average_precision_score(y_true, y_probability),
            "ROC_AUC": roc_auc_score(y_true, y_probability),
            "FalsePositives": fp,
            "FalseNegatives": fn,
        })

model_1_results = pd.DataFrame(model_1_results)
model_1_percentage_columns = [
    "ChurnRate", "ContactRate", "Precision", "Recall",
    "Specificity", "F1", "Accuracy", "BalancedAccuracy",
    "PR_AUC", "ROC_AUC",
]
model_1_display = model_1_results.copy()
model_1_display[model_1_percentage_columns] *= 100
model_1_display.round(2)

### 7.1 Two probability-ranking metrics

`ROC_AUC` measures how often a randomly selected future churner receives a higher probability than a randomly selected non-churner. A value of 50% corresponds to random ranking and 100% to perfect ranking.

`PR_AUC`, implemented here as average precision, focuses on the positive class. Its naive reference level is approximately the churn rate of the evaluated period. It is therefore interpreted relative to `ChurnRate`, not relative to 50%. Both metrics evaluate ranking across all possible thresholds and do not depend on the temporary 0.50 classification threshold.

### 7.2 Learned probability curves

The following figure shows the probability learned by each walk-forward fold for recency values from 0 to 60 days. The curves are learned from different historical periods, so a visible difference indicates temporal instability in the relationship between recency and churn.

In [ ]:
import matplotlib.pyplot as plt

recency_grid = pd.DataFrame({
    "RecencyDays": range(61)
})
plt.figure(figsize=(9, 5))
for (approach, fold_name), model in model_1_fitted.items():
    if approach != "B":
        continue
    probabilities = model.predict_proba(recency_grid)[:, 1]
    plt.plot(
        recency_grid["RecencyDays"],
        probabilities,
        label=fold_name,
    )
plt.axhline(0.50, color="grey", linestyle="--", label="0.50 threshold")
plt.axvline(30, color="black", linestyle=":", label="30-day baseline")
plt.title("Model 1: learned churn probability from recency")
plt.xlabel("RecencyDays")
plt.ylabel("Predicted probability of churn in the next 30 days")
plt.ylim(0, 1)
plt.legend()
plt.tight_layout()
plt.show()

learned_recency_cutoffs = []
fine_recency_grid = pd.DataFrame({
    "RecencyDays": [value / 100 for value in range(6001)]
})
for (approach, fold_name), model in model_1_fitted.items():
    probabilities = model.predict_proba(fine_recency_grid)[:, 1]
    learned_recency_cutoffs.append({
        "Approach": approach,
        "Fold": fold_name,
        "RecencyAtProbability50": fine_recency_grid.loc[
            probabilities >= 0.50, "RecencyDays"
        ].min(),
    })
pd.DataFrame(learned_recency_cutoffs)

## 8. Decision after Model 1

The model ranks customers strongly in both validation periods: ROC AUC is 96.50% on October-November and 97.31% on January-February. PR AUC is respectively 87.77% and 95.48%, well above each period's churn rate. Train and validation ROC AUC remain close, so there is no obvious overfitting signal with this single feature.

At the temporary 0.50 probability threshold, the model learns an effective recency cutoff near 34.61 days in fold 1 and 34.85 days in fold 2. Compared with the 30-day baseline, it reduces false positives from 284 to 182 in the first validation and from 149 to 113 in the second. The cost is substantial: false negatives rise from 4 to 178 and from 0 to 166. Recall falls to 81.32% and 88.42%. The default-threshold model therefore does not beat the baseline on F1 or balanced accuracy.

This does not invalidate logistic regression. The ranking is strong, while the 0.50 decision threshold is only a temporary convention. Model 2 will add only purchase frequency while keeping the same folds. This isolates whether one additional behavioral signal improves prediction beyond recency.

## 9. Model 2: recency plus purchase frequency

### Hypothesis

For two customers with the same current recency, the customer with more historical purchases is more likely to purchase again before reaching the churn deadline. `PurchaseFrequency` should therefore have a negative coefficient after recency is controlled.

This model changes only one thing relative to Model 1: `PurchaseFrequency` is added. The algorithm, scaling, folds, regularization, and temporary 0.50 threshold remain identical, making the comparison attributable to this new signal.

In [ ]:
model_2_features = ["RecencyDays", "PurchaseFrequency"]
model_2_results = []
model_2_coefficients = []
model_2_fitted = {}

for _, fold in model_selection_folds.iterrows():
    train, validation = get_fold_data(fold)
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, solver="liblinear"),
    )
    model.fit(train[model_2_features], train[target_column])
    model_2_fitted[(fold["Approach"], fold["Fold"])] = model

    coefficients = model.named_steps["logisticregression"].coef_[0]
    model_2_coefficients.append({
        "Approach": fold["Approach"],
        "Fold": fold["Fold"],
        "RecencyCoefficient": coefficients[0],
        "FrequencyCoefficient": coefficients[1],
    })

    for dataset_name, dataset in [("train", train), ("validation", validation)]:
        y_true = dataset[target_column]
        y_probability = model.predict_proba(dataset[model_2_features])[:, 1]
        y_pred = y_probability >= 0.50
        tn, fp, fn, tp = confusion_matrix(
            y_true, y_pred, labels=[False, True]
        ).ravel()

        model_2_results.append({
            "Approach": fold["Approach"],
            "Fold": fold["Fold"],
            "Dataset": dataset_name,
            "Rows": len(dataset),
            "ChurnRate": y_true.mean(),
            "ContactRate": y_pred.mean(),
            "Precision": precision_score(y_true, y_pred),
            "Recall": recall_score(y_true, y_pred),
            "Specificity": tn / (tn + fp),
            "F1": f1_score(y_true, y_pred),
            "Accuracy": accuracy_score(y_true, y_pred),
            "BalancedAccuracy": balanced_accuracy_score(y_true, y_pred),
            "PR_AUC": average_precision_score(y_true, y_probability),
            "ROC_AUC": roc_auc_score(y_true, y_probability),
            "FalsePositives": fp,
            "FalseNegatives": fn,
        })

model_2_results = pd.DataFrame(model_2_results)
model_2_coefficients = pd.DataFrame(model_2_coefficients)
model_2_display = model_2_results.copy()
model_2_display[model_1_percentage_columns] *= 100
display(model_2_coefficients.round(3))
model_2_display.round(2)

### 9.1 Validation comparison with Model 1

Only validation rows are compared. A useful new signal should improve ranking metrics such as PR AUC or ROC AUC consistently across both validation periods. Threshold-dependent changes in precision and recall are secondary at this stage because the 0.50 threshold remains temporary.

In [ ]:
comparison_columns = [
    "Approach", "Fold", "Precision", "Recall",
    "Specificity", "F1", "BalancedAccuracy",
    "PR_AUC", "ROC_AUC", "ContactRate",
    "FalsePositives", "FalseNegatives",
]
model_comparison = pd.concat([
    model_1_display.loc[
        model_1_display["Dataset"].eq("validation"), comparison_columns
    ].assign(Model="Model 1: Recency"),
    model_2_display.loc[
        model_2_display["Dataset"].eq("validation"), comparison_columns
    ].assign(Model="Model 2: Recency + Frequency"),
], ignore_index=True)
model_comparison[["Model"] + comparison_columns].round(2)

## 10. Decision after Model 2

The frequency coefficient is negative and stable in both temporal folds, approximately -1.13 and -1.20 after standardization. At the same recency, greater historical purchase frequency is therefore associated with lower predicted churn risk, as hypothesized.

Ranking improves in both validation periods. PR AUC rises from 87.77% to 89.05% in fold 1 and from 95.48% to 96.34% in fold 2. ROC AUC rises from 96.50% to 96.69% and from 97.31% to 97.61%. Precision also increases while false positives fall from 182 to 156 and from 113 to 84.

At the temporary 0.50 threshold, recall is almost unchanged in fold 1 but falls from 88.42% to 84.93% in fold 2. This threshold-dependent tradeoff will be handled later. Because frequency adds consistent ranking information beyond recency, `PurchaseFrequency` is retained for the next model.

## 11. Model 3: adding monetary value

### Hypothesis

At the same recency and purchase frequency, a customer who has historically spent more may have a stronger commercial relationship with the retailer and therefore a lower churn risk. `MonetaryValue` should add information beyond recency and frequency.

Only `MonetaryValue` is added. The algorithm, scaling, two folds of approach B, regularization, and temporary 0.50 threshold remain unchanged.

In [ ]:
model_3_features = ["RecencyDays", "PurchaseFrequency", "MonetaryValue"]
model_3_results = []
model_3_coefficients = []

for _, fold in model_selection_folds.loc[model_selection_folds["Approach"].eq("B")].iterrows():
    train, validation = get_fold_data(fold)
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, solver="liblinear"),
    )
    model.fit(train[model_3_features], train[target_column])

    coefficients = model.named_steps["logisticregression"].coef_[0]
    model_3_coefficients.append({
        "Fold": fold["Fold"],
        "RecencyCoefficient": coefficients[0],
        "FrequencyCoefficient": coefficients[1],
        "MonetaryCoefficient": coefficients[2],
    })

    y_true = validation[target_column]
    y_probability = model.predict_proba(validation[model_3_features])[:, 1]
    y_pred = y_probability >= 0.50
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[False, True]
    ).ravel()

    model_3_results.append({
        "Fold": fold["Fold"],
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "PR_AUC": average_precision_score(y_true, y_probability),
        "ROC_AUC": roc_auc_score(y_true, y_probability),
        "FalsePositives": fp,
        "FalseNegatives": fn,
    })

model_3_results = pd.DataFrame(model_3_results)
model_3_coefficients = pd.DataFrame(model_3_coefficients)
model_3_display = model_3_results.copy()
model_3_display[["Precision", "Recall", "PR_AUC", "ROC_AUC"]] *= 100
display(model_3_coefficients.round(3))
model_3_display.round(2)

### 11.1 Does monetary value improve the model?

The comparison is restricted to the two validation folds of approach B. Model 2 is the reference. We first examine whether PR AUC improves consistently before interpreting the temporary 0.50 threshold.

In [ ]:
model_2_b = model_2_display.loc[
    model_2_display["Approach"].eq("B")
    & model_2_display["Dataset"].eq("validation"),
    ["Fold", "Precision", "Recall", "PR_AUC", "ROC_AUC",
     "FalsePositives", "FalseNegatives"],
].assign(Model="Model 2: R + F")

model_3_b = model_3_display.assign(Model="Model 3: R + F + M")
pd.concat([model_2_b, model_3_b], ignore_index=True)[
    ["Model", "Fold", "Precision", "Recall", "PR_AUC",
     "ROC_AUC", "FalsePositives", "FalseNegatives"]
].round(2)

## 12. Decision after Model 3

Adding raw `MonetaryValue` does not improve ranking. PR AUC remains 89.05% in fold 1 and 96.34% in fold 2, while ROC AUC is also unchanged at the displayed precision. False positives remain 156 and 84.

The standardized monetary coefficient is -0.266 in fold 1 but only -0.027 in fold 2. Its direction is negative, but its magnitude is not temporally stable. Most of the information carried by cumulative spending is probably already represented by purchase frequency, and the raw monetary distribution is highly skewed.

Therefore, raw `MonetaryValue` is not retained in the current model. A logarithmic monetary transformation may be tested later as a separate hypothesis, but it should not be introduced automatically. Model 2 remains the current reference model.

## 13. Model 4: adding average invoice value

### Hypothesis

At the same recency and purchase frequency, the average amount spent per invoice may distinguish low-value occasional purchases from a stronger commercial relationship. `AverageInvoiceValue` may therefore add information that cumulative `MonetaryValue` did not provide.

This model adds only `AverageInvoiceValue` to the retained recency and frequency features. Raw `MonetaryValue` remains excluded. The two folds of approach B and all model parameters remain unchanged.

In [ ]:
model_4_features = ["RecencyDays", "PurchaseFrequency", "AverageInvoiceValue"]
model_4_results = []
model_4_coefficients = []

for _, fold in model_selection_folds.loc[model_selection_folds["Approach"].eq("B")].iterrows():
    train, validation = get_fold_data(fold)
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, solver="liblinear"),
    )
    model.fit(train[model_4_features], train[target_column])

    coefficients = model.named_steps["logisticregression"].coef_[0]
    model_4_coefficients.append({
        "Fold": fold["Fold"],
        "RecencyCoefficient": coefficients[0],
        "FrequencyCoefficient": coefficients[1],
        "AverageInvoiceValueCoefficient": coefficients[2],
    })

    y_true = validation[target_column]
    y_probability = model.predict_proba(validation[model_4_features])[:, 1]
    y_pred = y_probability >= 0.50
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[False, True]
    ).ravel()

    model_4_results.append({
        "Fold": fold["Fold"],
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "PR_AUC": average_precision_score(y_true, y_probability),
        "ROC_AUC": roc_auc_score(y_true, y_probability),
        "FalsePositives": fp,
        "FalseNegatives": fn,
    })

model_4_results = pd.DataFrame(model_4_results)
model_4_coefficients = pd.DataFrame(model_4_coefficients)
model_4_display = model_4_results.copy()
model_4_display[["Precision", "Recall", "PR_AUC", "ROC_AUC"]] *= 100
display(model_4_coefficients.round(3))
model_4_display.round(2)

### 13.1 Does average invoice value improve the model?

Model 4 is compared with Model 2 on the two validation folds of approach B. The primary criterion remains a consistent improvement in PR AUC.

In [ ]:
model_4_b = model_4_display.assign(Model="Model 4: R + F + Average invoice")
pd.concat([model_2_b, model_4_b], ignore_index=True)[
    ["Model", "Fold", "Precision", "Recall", "PR_AUC",
     "ROC_AUC", "FalsePositives", "FalseNegatives"]
].round(2)

## 14. Decision after Model 4

`AverageInvoiceValue` does not provide a meaningful improvement beyond recency and frequency. PR AUC changes only from 89.05% to 89.06% in fold 1 and from 96.34% to 96.35% in fold 2. ROC AUC is unchanged at the displayed precision.

Its standardized coefficient is also small, -0.060 in fold 1 and -0.024 in fold 2. The negative direction suggests a slightly lower estimated churn risk for customers with larger average invoices, but the effect is too weak to justify keeping the feature.

Therefore, `AverageInvoiceValue` is not retained. Model 2, using `RecencyDays` and `PurchaseFrequency`, remains the reference model.

## 15. Model 5: adding relative recency

### Hypothesis

The same absolute recency can have different meanings for different customers. Thirty inactive days are unusual for a weekly buyer but normal for a customer who typically purchases every two months. `RecencyToMedianGapRatio` compares current recency with the customer's own median historical interval.

The ratio is missing until two distinct historical purchase days are available. Those values are replaced by the training-fold median inside the pipeline. This preserves all customers and prevents information from validation from influencing imputation. No missingness indicator is added yet, so this experiment introduces only one new predictive signal.

In [ ]:
from sklearn.impute import SimpleImputer

model_5_features = [
    "RecencyDays", "PurchaseFrequency", "RecencyToMedianGapRatio"
]
model_5_results = []
model_5_coefficients = []

for _, fold in model_selection_folds.loc[model_selection_folds["Approach"].eq("B")].iterrows():
    train, validation = get_fold_data(fold)
    model = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(max_iter=1000, solver="liblinear"),
    )
    model.fit(train[model_5_features], train[target_column])

    coefficients = model.named_steps["logisticregression"].coef_[0]
    model_5_coefficients.append({
        "Fold": fold["Fold"],
        "RecencyCoefficient": coefficients[0],
        "FrequencyCoefficient": coefficients[1],
        "RecencyRatioCoefficient": coefficients[2],
    })

    y_true = validation[target_column]
    y_probability = model.predict_proba(validation[model_5_features])[:, 1]
    y_pred = y_probability >= 0.50
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[False, True]
    ).ravel()

    model_5_results.append({
        "Fold": fold["Fold"],
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "PR_AUC": average_precision_score(y_true, y_probability),
        "ROC_AUC": roc_auc_score(y_true, y_probability),
        "FalsePositives": fp,
        "FalseNegatives": fn,
    })

model_5_results = pd.DataFrame(model_5_results)
model_5_coefficients = pd.DataFrame(model_5_coefficients)
model_5_display = model_5_results.copy()
model_5_display[["Precision", "Recall", "PR_AUC", "ROC_AUC"]] *= 100
display(model_5_coefficients.round(3))
model_5_display.round(2)

### 15.1 Does relative recency improve the model?

Model 5 is compared with the retained Model 2 on both validation folds of approach B. The ratio is retained only if it adds stable ranking information beyond absolute recency and frequency.

In [ ]:
model_5_b = model_5_display.assign(Model="Model 5: R + F + Recency ratio")
pd.concat([model_2_b, model_5_b], ignore_index=True)[
    ["Model", "Fold", "Precision", "Recall", "PR_AUC",
     "ROC_AUC", "FalsePositives", "FalseNegatives"]
].round(2)

## 16. Decision after Model 5

The raw relative-recency ratio does not improve ranking beyond absolute recency and frequency. PR AUC remains 89.05% in fold 1 and changes from 96.34% to 96.32% in fold 2. ROC AUC is effectively unchanged.

Its standardized coefficient is only 0.039 in fold 1 and 0.020 in fold 2. The direction is consistent but the effect is negligible after absolute recency and purchase frequency are controlled. At the temporary 0.50 threshold, false positives fall by only one customer in each fold while false negatives increase by two.

Therefore, raw `RecencyToMedianGapRatio` is not retained in the current logistic model. The feature remains available for a later non-linear model, which may use customer rhythm through thresholds or interactions that a linear log-odds model cannot represent. Model 2 remains the reference model.

### 15.2 Sensitivity check: mean imputation

The previous experiment used the training-fold median for customers without two observed purchase days. This sensitivity check replaces those missing ratios with the training-fold mean instead. The folds, features, scaling, model parameters, and decision threshold remain unchanged.

In [ ]:
model_5_mean_results = []

for _, fold in model_selection_folds.loc[model_selection_folds["Approach"].eq("B")].iterrows():
    train, validation = get_fold_data(fold)
    model = make_pipeline(
        SimpleImputer(strategy="mean"),
        StandardScaler(),
        LogisticRegression(max_iter=1000, solver="liblinear"),
    )
    model.fit(train[model_5_features], train[target_column])

    y_true = validation[target_column]
    y_probability = model.predict_proba(validation[model_5_features])[:, 1]
    y_pred = y_probability >= 0.50
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[False, True]
    ).ravel()

    model_5_mean_results.append({
        "Fold": fold["Fold"],
        "ImputedRatio": model.named_steps["simpleimputer"].statistics_[2],
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "PR_AUC": average_precision_score(y_true, y_probability),
        "ROC_AUC": roc_auc_score(y_true, y_probability),
        "FalsePositives": fp,
        "FalseNegatives": fn,
    })

model_5_mean_results = pd.DataFrame(model_5_mean_results)
model_5_mean_display = model_5_mean_results.copy()
model_5_mean_display[["Precision", "Recall", "PR_AUC", "ROC_AUC"]] *= 100
model_5_mean_display.round(2)

### 15.3 Median versus mean imputation

In [ ]:
median_imputation = model_5_display.assign(Imputation="Median")
mean_imputation = model_5_mean_display.assign(Imputation="Mean")
pd.concat([median_imputation, mean_imputation], ignore_index=True)[
    ["Imputation", "Fold", "Precision", "Recall", "PR_AUC",
     "ROC_AUC", "FalsePositives", "FalseNegatives"]
].round(2)

### Sensitivity-check conclusion

Mean imputation does not change the conclusion. PR AUC is 89.06% in fold 1 and 96.32% in fold 2, compared with 89.05% and 96.32% under median imputation. False positives are identical, while mean imputation produces two additional false negatives across both folds.

The training-fold means, approximately 0.79 and 0.71, are notably higher than the medians, approximately 0.45 and 0.39, because the ratio is right-skewed. Despite that difference, predictive performance is effectively unchanged. The ratio remains excluded from the current logistic model.

## 17. Model 6: adding Christmas purchase concentration

### Hypothesis

Customers whose historical purchases concentrate on Christmas products may follow a more seasonal return pattern than customers buying general products. `ChristmasInvoiceShare` is the proportion of valid historical invoices containing at least one product matched by the conservative Christmas category.

This experiment adds only that share to the retained recency and frequency features. The two validation folds of approach B, scaling, logistic-regression parameters, and temporary 0.50 threshold remain unchanged.

In [ ]:
model_6_features = [
    "RecencyDays", "PurchaseFrequency", "ChristmasInvoiceShare"
]
model_6_results = []
model_6_coefficients = []

for _, fold in model_selection_folds.loc[model_selection_folds["Approach"].eq("B")].iterrows():
    train, validation = get_fold_data(fold)
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, solver="liblinear"),
    )
    model.fit(train[model_6_features], train[target_column])

    coefficients = model.named_steps["logisticregression"].coef_[0]
    model_6_coefficients.append({
        "Fold": fold["Fold"],
        "RecencyCoefficient": coefficients[0],
        "FrequencyCoefficient": coefficients[1],
        "ChristmasShareCoefficient": coefficients[2],
    })

    y_true = validation[target_column]
    y_probability = model.predict_proba(validation[model_6_features])[:, 1]
    y_pred = y_probability >= 0.50
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[False, True]
    ).ravel()

    model_6_results.append({
        "Fold": fold["Fold"],
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "PR_AUC": average_precision_score(y_true, y_probability),
        "ROC_AUC": roc_auc_score(y_true, y_probability),
        "FalsePositives": fp,
        "FalseNegatives": fn,
    })

model_6_results = pd.DataFrame(model_6_results)
model_6_coefficients = pd.DataFrame(model_6_coefficients)
model_6_display = model_6_results.copy()
model_6_display[["Precision", "Recall", "PR_AUC", "ROC_AUC"]] *= 100
display(model_6_coefficients.round(3))
model_6_display.round(2)

### 17.1 Does Christmas purchase concentration improve the model?

In [ ]:
model_6_b = model_6_display.assign(Model="Model 6: R + F + Christmas share")
pd.concat([model_2_b, model_6_b], ignore_index=True)[
    ["Model", "Fold", "Precision", "Recall", "PR_AUC",
     "ROC_AUC", "FalsePositives", "FalseNegatives"]
].round(2)

## 18. Decision after Model 6

As a standalone linear effect, `ChristmasInvoiceShare` adds almost no ranking information. PR AUC changes from 89.05% to 89.06% in fold 1 and from 96.34% to 96.36% in fold 2. ROC AUC is effectively unchanged.

The standardized coefficient is 0.004 in fold 1 and 0.081 in fold 2, so its magnitude is not stable across time. At the temporary 0.50 threshold, fold 1 is unchanged apart from one additional false negative. Fold 2 has 18 fewer false negatives but one additional false positive, without a meaningful ranking gain.

Therefore, `ChristmasInvoiceShare` is not retained as a standalone feature in the current logistic model. The result does not rule out seasonality: Christmas concentration may matter only through an interaction with the reference month, which should be treated as a separate future hypothesis. Model 2 remains the reference model.

## 19. Model 7: adding product breadth

### Hypothesis

For the same number of historical invoices, a customer who has purchased more distinct products may have a broader relationship with the retailer and a lower churn risk. `UniqueProductsPurchased` counts distinct physical `StockCode` values available in the valid history at each reference date.

This experiment adds only product breadth to the retained recency and frequency features. The two folds of approach B and all model parameters remain unchanged.

In [ ]:
model_7_features = [
    "RecencyDays", "PurchaseFrequency", "UniqueProductsPurchased"
]
model_7_results = []
model_7_coefficients = []

for _, fold in model_selection_folds.loc[model_selection_folds["Approach"].eq("B")].iterrows():
    train, validation = get_fold_data(fold)
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, solver="liblinear"),
    )
    model.fit(train[model_7_features], train[target_column])

    coefficients = model.named_steps["logisticregression"].coef_[0]
    model_7_coefficients.append({
        "Fold": fold["Fold"],
        "RecencyCoefficient": coefficients[0],
        "FrequencyCoefficient": coefficients[1],
        "UniqueProductsCoefficient": coefficients[2],
    })

    y_true = validation[target_column]
    y_probability = model.predict_proba(validation[model_7_features])[:, 1]
    y_pred = y_probability >= 0.50
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[False, True]
    ).ravel()

    model_7_results.append({
        "Fold": fold["Fold"],
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "PR_AUC": average_precision_score(y_true, y_probability),
        "ROC_AUC": roc_auc_score(y_true, y_probability),
        "FalsePositives": fp,
        "FalseNegatives": fn,
    })

model_7_results = pd.DataFrame(model_7_results)
model_7_coefficients = pd.DataFrame(model_7_coefficients)
model_7_display = model_7_results.copy()
model_7_display[["Precision", "Recall", "PR_AUC", "ROC_AUC"]] *= 100
display(model_7_coefficients.round(3))
model_7_display.round(2)

### 19.1 Does product breadth improve the model?

In [ ]:
model_7_b = model_7_display.assign(Model="Model 7: R + F + Unique products")
pd.concat([model_2_b, model_7_b], ignore_index=True)[
    ["Model", "Fold", "Precision", "Recall", "PR_AUC",
     "ROC_AUC", "FalsePositives", "FalseNegatives"]
].round(2)

## 20. Decision after Model 7

`UniqueProductsPurchased` has a stable negative standardized coefficient, -0.208 in fold 1 and -0.191 in fold 2. After purchase frequency is controlled, greater product breadth is therefore associated with slightly lower predicted churn risk.

However, predictive improvement is not consistent. PR AUC rises from 89.05% to 89.12% in fold 1 but falls from 96.34% to 96.31% in fold 2. At the temporary 0.50 threshold, recall also decreases in both periods. The feature is strongly correlated with purchase frequency, approximately 0.71 over the complete modeling table.

Therefore, the absolute distinct-product count is not retained in the current logistic model. The stable association justifies a separate future test of a frequency-normalized breadth measure, such as `UniqueProductsPurchased / PurchaseFrequency`, but that would be a new hypothesis. Model 2 remains the reference model.

## 21. Model 8: adding product breadth per invoice

### Hypothesis

`ProductsPerInvoice` divides cumulative distinct products by purchase frequency. It tests whether product exploration per historical invoice adds information that the absolute product count could not provide independently of frequency.

Only this ratio is added to the retained recency and frequency features. The two folds of approach B and all model parameters remain unchanged.

In [ ]:
model_8_features = [
    "RecencyDays", "PurchaseFrequency", "ProductsPerInvoice"
]
model_8_results = []
model_8_coefficients = []

for _, fold in model_selection_folds.loc[model_selection_folds["Approach"].eq("B")].iterrows():
    train, validation = get_fold_data(fold)
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, solver="liblinear"),
    )
    model.fit(train[model_8_features], train[target_column])

    coefficients = model.named_steps["logisticregression"].coef_[0]
    model_8_coefficients.append({
        "Fold": fold["Fold"],
        "RecencyCoefficient": coefficients[0],
        "FrequencyCoefficient": coefficients[1],
        "ProductsPerInvoiceCoefficient": coefficients[2],
    })

    y_true = validation[target_column]
    y_probability = model.predict_proba(validation[model_8_features])[:, 1]
    y_pred = y_probability >= 0.50
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[False, True]
    ).ravel()

    model_8_results.append({
        "Fold": fold["Fold"],
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "PR_AUC": average_precision_score(y_true, y_probability),
        "ROC_AUC": roc_auc_score(y_true, y_probability),
        "FalsePositives": fp,
        "FalseNegatives": fn,
    })

model_8_results = pd.DataFrame(model_8_results)
model_8_coefficients = pd.DataFrame(model_8_coefficients)
model_8_display = model_8_results.copy()
model_8_display[["Precision", "Recall", "PR_AUC", "ROC_AUC"]] *= 100
display(model_8_coefficients.round(3))
model_8_display.round(2)

### 21.1 Does normalized product breadth improve the model?

In [ ]:
model_8_b = model_8_display.assign(Model="Model 8: R + F + Products per invoice")
pd.concat([model_2_b, model_8_b], ignore_index=True)[
    ["Model", "Fold", "Precision", "Recall", "PR_AUC",
     "ROC_AUC", "FalsePositives", "FalseNegatives"]
].round(2)

## 22. Decision after Model 8

Normalizing product breadth by invoice frequency does not add predictive information. PR AUC changes only from 89.05% to 89.06% in fold 1 and remains 96.34% in fold 2. ROC AUC is effectively unchanged.

The standardized ratio coefficient is nearly zero in both periods, -0.021 and -0.013. At the temporary 0.50 threshold, fold 1 exchanges one false positive for one additional false negative, while fold 2 produces two additional false negatives with no reduction in false positives.

Therefore, `ProductsPerInvoice` is not retained. Product breadth has an interpretable descriptive association but does not improve this logistic churn model beyond recency and purchase frequency. Model 2 remains the reference model.

## 23. Model 9: adding observed customer tenure

### Hypothesis

At the same recency and purchase frequency, a customer observed over a longer relationship may have a different churn risk from a recently acquired customer. `ObservedTenureDays` measures the time from the first purchase visible in this dataset to the reference date.

This is observed tenure, not guaranteed lifetime tenure, because transactions before December 2009 are unavailable. Only this feature is added to the retained model, while the folds and all model parameters remain unchanged.

In [ ]:
model_9_features = [
    "RecencyDays", "PurchaseFrequency", "ObservedTenureDays"
]
model_9_results = []
model_9_coefficients = []

for _, fold in model_selection_folds.loc[model_selection_folds["Approach"].eq("B")].iterrows():
    train, validation = get_fold_data(fold)
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, solver="liblinear"),
    )
    model.fit(train[model_9_features], train[target_column])

    coefficients = model.named_steps["logisticregression"].coef_[0]
    model_9_coefficients.append({
        "Fold": fold["Fold"],
        "RecencyCoefficient": coefficients[0],
        "FrequencyCoefficient": coefficients[1],
        "ObservedTenureCoefficient": coefficients[2],
    })

    y_true = validation[target_column]
    y_probability = model.predict_proba(validation[model_9_features])[:, 1]
    y_pred = y_probability >= 0.50
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[False, True]
    ).ravel()

    model_9_results.append({
        "Fold": fold["Fold"],
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "PR_AUC": average_precision_score(y_true, y_probability),
        "ROC_AUC": roc_auc_score(y_true, y_probability),
        "FalsePositives": fp,
        "FalseNegatives": fn,
    })

model_9_results = pd.DataFrame(model_9_results)
model_9_coefficients = pd.DataFrame(model_9_coefficients)
model_9_display = model_9_results.copy()
model_9_display[["Precision", "Recall", "PR_AUC", "ROC_AUC"]] *= 100
display(model_9_coefficients.round(3))
model_9_display.round(2)

### 23.1 Does observed tenure improve the model?

In [ ]:
model_9_b = model_9_display.assign(Model="Model 9: R + F + Observed tenure")
pd.concat([model_2_b, model_9_b], ignore_index=True)[
    ["Model", "Fold", "Precision", "Recall", "PR_AUC",
     "ROC_AUC", "FalsePositives", "FalseNegatives"]
].round(2)

## 24. Decision after Model 9

Observed tenure has a stable negative standardized coefficient, -0.148 in fold 1 and -0.156 in fold 2. At the same recency and frequency, customers observed for longer are therefore assigned a slightly lower churn risk.

This association does not improve ranking. PR AUC remains 89.05% in fold 1 and falls from 96.34% to 96.25% in fold 2. At the temporary 0.50 threshold, false positives fall from 156 to 148 and from 84 to 80, but false negatives rise from 174 to 193 and from 216 to 231.

Therefore, `ObservedTenureDays` is not retained in the current model. It changes the threshold tradeoff without improving threshold-independent discrimination, and its interpretation is limited by left-censoring at the dataset start. Model 2 remains the reference model.

## 25. Model 10: adding the United Kingdom indicator

### Hypothesis

Customers outside the retailer's main United Kingdom market may have different purchasing constraints or return behavior. `IsUK` equals 1 when the latest valid historical invoice at the reference date is recorded in the United Kingdom and 0 otherwise.

Only this binary signal is added to the retained recency and frequency features. The two folds of approach B and all model parameters remain unchanged.

In [ ]:
model_10_features = ["RecencyDays", "PurchaseFrequency", "IsUK"]
model_10_results = []
model_10_coefficients = []

for _, fold in model_selection_folds.loc[model_selection_folds["Approach"].eq("B")].iterrows():
    train, validation = get_fold_data(fold)
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, solver="liblinear"),
    )
    model.fit(train[model_10_features], train[target_column])

    coefficients = model.named_steps["logisticregression"].coef_[0]
    model_10_coefficients.append({
        "Fold": fold["Fold"],
        "RecencyCoefficient": coefficients[0],
        "FrequencyCoefficient": coefficients[1],
        "IsUKCoefficient": coefficients[2],
    })

    y_true = validation[target_column]
    y_probability = model.predict_proba(validation[model_10_features])[:, 1]
    y_pred = y_probability >= 0.50
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[False, True]
    ).ravel()

    model_10_results.append({
        "Fold": fold["Fold"],
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "PR_AUC": average_precision_score(y_true, y_probability),
        "ROC_AUC": roc_auc_score(y_true, y_probability),
        "FalsePositives": fp,
        "FalseNegatives": fn,
    })

model_10_results = pd.DataFrame(model_10_results)
model_10_coefficients = pd.DataFrame(model_10_coefficients)
model_10_display = model_10_results.copy()
model_10_display[["Precision", "Recall", "PR_AUC", "ROC_AUC"]] *= 100
display(model_10_coefficients.round(3))
model_10_display.round(2)

### 25.1 Does the UK indicator improve the model?

In [ ]:
model_10_b = model_10_display.assign(Model="Model 10: R + F + IsUK")
pd.concat([model_2_b, model_10_b], ignore_index=True)[
    ["Model", "Fold", "Precision", "Recall", "PR_AUC",
     "ROC_AUC", "FalsePositives", "FalseNegatives"]
].round(2)

## 26. Decision after Model 10

`IsUK` does not improve predictive ranking. PR AUC changes only from 89.05% to 89.06% in fold 1 and from 96.34% to 96.32% in fold 2. ROC AUC is effectively unchanged.

The standardized coefficient is small and negative in both periods, -0.031 and -0.027. At the temporary 0.50 threshold, fold 1 keeps the same false-positive count and adds three false negatives. Fold 2 removes one false positive but adds two false negatives. About 91.34% of all modeling snapshots are UK observations, which also limits the information available from this split.

Therefore, `IsUK` is not retained. This confirms the earlier descriptive finding that the UK versus non-UK distinction does not materially separate churn risk in this dataset. Model 2 remains the reference model.

## 27. Model 11: adding previous cancellation history

### Hypothesis

A previous cancellation may capture dissatisfaction, purchasing complexity, or simply deeper engagement with the retailer. `HasCancellation` equals 1 when at least one cancellation invoice is known strictly before the reference date and 0 otherwise. Its direction is therefore an empirical question rather than a predetermined assumption.

Only this binary signal is added to recency and frequency. The two folds of approach B and all model parameters remain unchanged.

In [ ]:
model_11_features = ["RecencyDays", "PurchaseFrequency", "HasCancellation"]
model_11_results = []
model_11_coefficients = []

for _, fold in model_selection_folds.loc[model_selection_folds["Approach"].eq("B")].iterrows():
    train, validation = get_fold_data(fold)
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, solver="liblinear"),
    )
    model.fit(train[model_11_features], train[target_column])

    coefficients = model.named_steps["logisticregression"].coef_[0]
    model_11_coefficients.append({
        "Fold": fold["Fold"],
        "RecencyCoefficient": coefficients[0],
        "FrequencyCoefficient": coefficients[1],
        "HasCancellationCoefficient": coefficients[2],
    })

    y_true = validation[target_column]
    y_probability = model.predict_proba(validation[model_11_features])[:, 1]
    y_pred = y_probability >= 0.50
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[False, True]
    ).ravel()

    model_11_results.append({
        "Fold": fold["Fold"],
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "PR_AUC": average_precision_score(y_true, y_probability),
        "ROC_AUC": roc_auc_score(y_true, y_probability),
        "FalsePositives": fp,
        "FalseNegatives": fn,
    })

model_11_results = pd.DataFrame(model_11_results)
model_11_coefficients = pd.DataFrame(model_11_coefficients)
model_11_display = model_11_results.copy()
model_11_display[["Precision", "Recall", "PR_AUC", "ROC_AUC"]] *= 100
display(model_11_coefficients.round(3))
model_11_display.round(2)

### 27.1 Does previous cancellation history improve the model?

In [ ]:
model_11_b = model_11_display.assign(Model="Model 11: R + F + Cancellation history")
pd.concat([model_2_b, model_11_b], ignore_index=True)[
    ["Model", "Fold", "Precision", "Recall", "PR_AUC",
     "ROC_AUC", "FalsePositives", "FalseNegatives"]
].round(2)

## 28. Decision after Model 11

`HasCancellation` produces a small but consistent ranking improvement. PR AUC rises from 89.05% to 89.14% in fold 1 and from 96.34% to 96.37% in fold 2. The standardized coefficient is also stable, -0.151 and -0.144.

The negative direction means that, after recency and frequency are controlled, customers with a known cancellation history receive slightly lower churn probabilities. In this dataset, a cancellation appears to capture customer engagement or transaction complexity rather than acting purely as a dissatisfaction signal. This is an association, not a causal conclusion.

The gain is too small to call the feature strong, and threshold-dependent recall falls slightly. Nevertheless, because ranking improves in both periods and the coefficient is stable, `HasCancellation` is retained provisionally. Model 11 becomes the provisional reference model, subject to later parsimony checks and final threshold selection.

## 29. Model 12: adding recent purchase momentum

### Hypothesis

Two customers can have the same cumulative frequency but very different recent activity. `PurchasesLast30Days` counts valid invoices in `[ReferenceDate - 30 days, ReferenceDate)` and may identify customers whose purchasing momentum is still strong.

The feature also encodes part of the operational churn boundary: a customer with at least 30 days of recency necessarily has no valid purchase in this window. This is not leakage because the entire window precedes the reference date, but it must be acknowledged when interpreting a strong effect. Model 12 adds only this feature to the provisional Model 11.

In [ ]:
model_12_features = [
    "RecencyDays", "PurchaseFrequency",
    "HasCancellation", "PurchasesLast30Days",
]
model_12_results = []
model_12_coefficients = []

for _, fold in model_selection_folds.loc[model_selection_folds["Approach"].eq("B")].iterrows():
    train, validation = get_fold_data(fold)
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, solver="liblinear"),
    )
    model.fit(train[model_12_features], train[target_column])

    coefficients = model.named_steps["logisticregression"].coef_[0]
    model_12_coefficients.append({
        "Fold": fold["Fold"],
        "RecencyCoefficient": coefficients[0],
        "FrequencyCoefficient": coefficients[1],
        "HasCancellationCoefficient": coefficients[2],
        "RecentPurchasesCoefficient": coefficients[3],
    })

    y_true = validation[target_column]
    y_probability = model.predict_proba(validation[model_12_features])[:, 1]
    y_pred = y_probability >= 0.50
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[False, True]
    ).ravel()

    model_12_results.append({
        "Fold": fold["Fold"],
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "PR_AUC": average_precision_score(y_true, y_probability),
        "ROC_AUC": roc_auc_score(y_true, y_probability),
        "FalsePositives": fp,
        "FalseNegatives": fn,
    })

model_12_results = pd.DataFrame(model_12_results)
model_12_coefficients = pd.DataFrame(model_12_coefficients)
model_12_display = model_12_results.copy()
model_12_display[["Precision", "Recall", "PR_AUC", "ROC_AUC"]] *= 100
display(model_12_coefficients.round(3))
model_12_display.round(2)

### 29.1 Does recent purchasing improve the model?

In [ ]:
model_12_b = model_12_display.assign(Model="Model 12: + Purchases last 30 days")
pd.concat([model_11_b, model_12_b], ignore_index=True)[
    ["Model", "Fold", "Precision", "Recall", "PR_AUC",
     "ROC_AUC", "FalsePositives", "FalseNegatives"]
].round(2)

## 30. Decision after Model 12

`PurchasesLast30Days` provides the clearest improvement since purchase frequency. PR AUC rises from 89.14% to 90.31% in fold 1 and from 96.37% to 97.10% in fold 2. ROC AUC rises from 96.69% to 97.10% and from 97.62% to 98.16%.

At the temporary 0.50 threshold, recall rises from 80.90% to 96.33% in fold 1 and from 84.44% to 94.49% in fold 2. False negatives fall from 182 to 35 and from 223 to 79. Precision decreases and false positives rise, so the operational threshold still requires later tuning.

The standardized recent-purchase coefficient is strongly negative and stable, -6.209 and -6.754. Part of this strength is structural because the 30-day history window aligns with the point at which an active customer can enter the 30-day churn-warning horizon. The feature is valid and available at scoring time, but it should be interpreted as both recent momentum and a non-linear encoding of the business boundary.

`PurchasesLast30Days` is retained. Model 12 becomes the new reference model with `RecencyDays`, `PurchaseFrequency`, `HasCancellation`, and `PurchasesLast30Days`.

## 31. Model 13: explicit churn-risk window

### Hypothesis

The gain attributed to `PurchasesLast30Days` may come primarily from separating customers below and above the 30-day recency boundary. `IsInChurnRiskWindow` equals 1 when `RecencyDays >= 30`. Model 13 replaces the recent purchase count with this explicit indicator while keeping all other retained features unchanged.

If performance remains similar, the simpler binary boundary should be preferred because it expresses the operational logic directly.

In [ ]:
model_13_features = [
    "RecencyDays", "PurchaseFrequency",
    "HasCancellation", "IsInChurnRiskWindow",
]
model_13_results = []
model_13_coefficients = []

for _, fold in model_selection_folds.loc[model_selection_folds["Approach"].eq("B")].iterrows():
    train, validation = get_fold_data(fold)
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, solver="liblinear"),
    )
    model.fit(train[model_13_features], train[target_column])

    coefficients = model.named_steps["logisticregression"].coef_[0]
    model_13_coefficients.append({
        "Fold": fold["Fold"],
        "RecencyCoefficient": coefficients[0],
        "FrequencyCoefficient": coefficients[1],
        "HasCancellationCoefficient": coefficients[2],
        "RiskWindowCoefficient": coefficients[3],
    })

    y_true = validation[target_column]
    y_probability = model.predict_proba(validation[model_13_features])[:, 1]
    y_pred = y_probability >= 0.50
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[False, True]
    ).ravel()

    model_13_results.append({
        "Fold": fold["Fold"],
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "PR_AUC": average_precision_score(y_true, y_probability),
        "ROC_AUC": roc_auc_score(y_true, y_probability),
        "FalsePositives": fp,
        "FalseNegatives": fn,
    })

model_13_results = pd.DataFrame(model_13_results)
model_13_coefficients = pd.DataFrame(model_13_coefficients)
model_13_display = model_13_results.copy()
model_13_display[["Precision", "Recall", "PR_AUC", "ROC_AUC"]] *= 100
display(model_13_coefficients.round(3))
model_13_display.round(2)

### 31.1 Recent purchase count versus explicit risk window

In [ ]:
model_13_b = model_13_display.assign(Model="Model 13: + Explicit risk window")
pd.concat([model_12_b, model_13_b], ignore_index=True)[
    ["Model", "Fold", "Precision", "Recall", "PR_AUC",
     "ROC_AUC", "FalsePositives", "FalseNegatives"]
].round(2)

## 32. Decision after Model 13

The explicit risk-window indicator reproduces the recent-purchase model. In fold 1, both models have 80.39% precision, 96.33% recall, 224 false positives, and 35 false negatives. PR AUC changes only from 90.31% to 90.33%.

In fold 2, PR AUC remains 97.10%. The risk-window model slightly increases recall from 94.49% to 94.91%, reducing false negatives from 79 to 73 while increasing false positives from 107 to 109.

This confirms that the strong gain came primarily from representing the structural 30-day boundary, not from the exact number of recent invoices. `IsInChurnRiskWindow` is preferred because it is simpler and directly interpretable. `PurchasesLast30Days` is removed from the retained set. Model 13 becomes the reference model with `RecencyDays`, `PurchaseFrequency`, `HasCancellation`, and `IsInChurnRiskWindow`.